# PelicanFS

PelicanFS is a filesystem specification (fsspec) implementation for the Pelican Platform. It provides a Python interface to interact with Pelican federations, allowing you to read, write, and manage objects across distributed storage systems.

**Installation:**
```bash
pip install pelicanfs
```

**The `pelicanfs` package comes pre-installed in the Guest OSPool Notebook!**

For comprehensive documentation, see the [PelicanFS README](https://github.com/PelicanPlatform/pelicanfs).

## Setup

There are three ways to use PelicanFS:

1. **PelicanFileSystem** - Connect to any Pelican federation by providing its discovery URL
2. **OSDFFileSystem** - Convenience class that automatically connects to OSDF (osg-htc.org)
3. **fsspec directly** - Use the `osdf://` or `pelican://` schemes with fsspec

Here, we instantiate three objects to demonstrate the use, which we will use for the rest of the tutorial. All three use the OSDF Federation, which has the Federation URL of `osg-htc.org`:

In [ ]:
from pelicanfs import PelicanFileSystem, OSDFFileSystem
import fsspec

# Option 1: PelicanFileSystem with explicit federation URL
pelfs = PelicanFileSystem('pelican://osg-htc.org')

# Option 2: OSDFFileSystem (equivalent to above for OSDF)
osdf = OSDFFileSystem()

# Option 3: fsspec with scheme
fs = fsspec.filesystem('osdf')

These file system objects have methods for interacting with objects via the Federation.
Usage typically looks like

```python
pelfs.<method>('<namespace_prefix>/<object_name>')
```

## Listing objects

To list the objects associated with this namespace using `pelicanfs`, use the `ls` method of the `pelfs` object we instantiated above:

In [ ]:
ls_results = pelfs.ls('/pelicanplatform/test')
print(ls_results)

In [ ]:
print([i['name'] for i in ls_results])

## Pattern Matching with Glob

Use `glob()` to find objects matching a pattern:

> **Note:** Glob operations with `**` patterns can be expensive for large namespaces. Consider using `maxdepth` to limit the search.

In [ ]:
txt_files = pelfs.glob('/pelicanplatform/test/*.txt')
print(txt_files)

## Reading Objects

There are several ways to read object contents:

**Using `cat()`** - Read contents directly into memory as bytes:

In [ ]:
content = pelfs.cat('/pelicanplatform/test/hello-world.txt')
print(content.decode('utf-8'))

**Using `open()`** - Get a file-like object for reading:

In [ ]:
with pelfs.open('/pelicanplatform/test/hello-world.txt', 'r') as f:
    print(f.read())

You can also use `fsspec.open()` directly as long as you pass in the correct scheme:

In [ ]:
with fsspec.open('osdf:///pelicanplatform/test/hello-world.txt', 'r') as f:
    print(f.read())

## Downloading Objects

Use `get()` to download objects to local files.

> **Note:** `cat()` and `open()` load data into memory for processing. `get()` saves objects as local files for persistent storage. When using `open()`, fsspec may store objects temporarily and clean them up when Python exits.

In [ ]:
pelfs.get('/pelicanplatform/test/hello-world.txt', 'hello-world.txt')

In [ ]:
with open('hello-world.txt', 'r') as f:
    my_file = f.read()

print(my_file)

You can combine the above steps using the `open` method of the `PelicanFileSystem` object:

In [ ]:
with pelfs.open('/pelicanplatform/test/hello-world.txt', 'r') as f:
    direct_read = f.read()

print(direct_read)

## Automated use of PelicanFS

Many Python packages work with `fsspec` automatically, which in turn means that they can work with `pelicanfs` automatically.
That means that you don't necessarily have to manually invoke `pelicanfs` to use it!

The included script `autoload.py` does just this:

In [ ]:
%%bash
cat autoload.py

In [ ]:
%%bash
python3 ./autoload.py

## Authorization

PelicanFS supports token-based authorization for accessing protected namespaces and performing write operations.

### Providing a Token via Headers

You can explicitly provide an authorization token when creating the filesystem:

In [ ]:
# Example: Using a Bearer token with PelicanFileSystem
# pelfs_auth = PelicanFileSystem(
#     'pelican://osg-htc.org',
#     headers={'Authorization': 'Bearer YOUR_TOKEN_HERE'}
# )

# Example: Using a Bearer token with fsspec
# fs_auth = fsspec.filesystem('osdf', headers={'Authorization': 'Bearer YOUR_TOKEN_HERE'})

### Environment Variables

PelicanFS automatically discovers tokens from environment variables:

- `BEARER_TOKEN` - Direct token value
- `BEARER_TOKEN_FILE` - Path to a file containing the token

### OAuth in Jupyter Notebooks

When using OAuth authentication in Jupyter notebooks, you need special handling for password input because Jupyter handles stdin differently than a terminal:

In [ ]:
import getpass
import sys
from io import StringIO

# Capture the password securely
password = getpass.getpass("Enter password to encrypt credentials file: ")

# Redirect stdin to provide the password
old_stdin = sys.stdin
sys.stdin = StringIO(password + "\n")

try:
    # Replace with your protected namespace path
    result = pelfs.ls("/your/protected/namespace/path")
    print(result)
finally:
    sys.stdin = old_stdin

### Writing Objects

To upload local files as objects, you need proper authorization:

In [ ]:
# Example: Upload a file (requires authorization)
# pelfs_auth.put('/local/path/file.txt', '/namespace/remote/path/object.txt')

# Using fsspec
# fs_auth.put('/local/path/file.txt', '/namespace/remote/path/object.txt')

# Upload multiple files recursively
# pelfs_auth.put('/local/directory/', '/namespace/remote/path/', recursive=True)

## Limitations

PelicanFS does not support the following operations (they will raise `NotImplementedError`):

- `rm` (remove objects)
- `cp` (copy objects within the federation)
- `mkdir` / `makedirs` (create collections)
- `open()` with write modes - use `put()` instead

## Next Steps

For more advanced usage, see:

- **[PelicanFS README](https://github.com/PelicanPlatform/pelicanfs)** - Full documentation including xarray, PyTorch, and pandas integrations